# 03 — Modelling
IEEE-CIS Fraud Detection

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score

# Load processed features
df = pd.read_parquet('../data/processed/features.parquet')

In [ ]:
# Split features and target
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# Class imbalance ratio for scale_pos_weight
ratio = (y == 0).sum() / (y == 1).sum()
print(f'scale_pos_weight: {ratio:.2f}')

In [ ]:
# Time-based cross validation (do NOT use random KFold)
tscv = TimeSeriesSplit(n_splits=5)
auc_scores  = []
prauc_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(
        scale_pos_weight=ratio,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        eval_metric='aucpr',
        random_state=42
    )
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)

    probs = model.predict_proba(X_val)[:, 1]
    auc   = roc_auc_score(y_val, probs)
    prauc = average_precision_score(y_val, probs)
    auc_scores.append(auc)
    prauc_scores.append(prauc)
    print(f'Fold {fold+1} — ROC-AUC: {auc:.4f} | PR-AUC: {prauc:.4f}')

print(f'\nMean ROC-AUC: {np.mean(auc_scores):.4f}')
print(f'Mean PR-AUC:  {np.mean(prauc_scores):.4f}')

In [ ]:
# Train final model on full data and log with MLflow
with mlflow.start_run():
    final_model = xgb.XGBClassifier(
        scale_pos_weight=ratio,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        eval_metric='aucpr',
        random_state=42
    )
    final_model.fit(X, y)

    mlflow.log_params({'n_estimators': 500, 'max_depth': 6, 'lr': 0.05})
    mlflow.log_metrics({'mean_roc_auc': np.mean(auc_scores),
                        'mean_pr_auc':  np.mean(prauc_scores)})
    mlflow.xgboost.log_model(final_model, 'model')
    final_model.save_model('../models/xgb_fraud.bin')
    print('Model saved to models/xgb_fraud.bin')